In [ ]:
# Cell 1: Install Libraries
# -------------------------
# Installs the necessary libraries for Reddit interaction, AI, UI, and data handling.
# - praw: For interacting with the Reddit API.
# - google-generativeai: For interacting with the Google Gemini API.
# - gradio: For creating the web user interface.
# - pandas: For handling and displaying data in tables (specifically the search results).
# The '!' prefix runs shell commands directly from the notebook.

!pip install praw google-generativeai gradio pandas python-dotenv
print("✅ Required Libraries Installed Successfully!")

In [ ]:
# Cell 2: Imports and Multi-Environment Secret Loading
# ----------------------------------------------------
# Imports necessary libraries and securely loads API keys/credentials
# based on the execution environment (Kaggle, Colab, or Local).

import os                   # For accessing environment variables (local fallback)
import praw                 # Reddit API wrapper
import google.generativeai as genai # Google Gemini API client
import gradio as gr         # Web UI framework
import pandas as pd         # Data manipulation (for tables)
from urllib.parse import urlparse # For parsing URLs (extracting post ID)
import warnings             # For suppressing unwanted warnings
from dotenv import load_dotenv # For loading .env file locally

# --- Configuration ---
# Suppress specific warnings if needed (optional, uncomment if noisy)
# warnings.filterwarnings("ignore", message="Using `tqdm.autonotebook.tqdm`")

print("🔐 Importing Secrets...")

# --- Environment Detection & Secret Loading ---

# Define the secrets we need
REQUIRED_SECRETS = [
    "GOOGLE_API_KEY",
    "REDDIT_CLIENT_ID",
    "REDDIT_CLIENT_SECRET",
    "REDDIT_USER_AGENT"
    # Add "REDDIT_USERNAME", "REDDIT_PASSWORD" here if using password auth version
]

# Dictionary to store loaded secrets
secrets = {key: None for key in REQUIRED_SECRETS}
secrets_loaded_successfully = False
secret_source = "None" # Track where secrets were loaded from

# --- Attempt 1: Kaggle Secrets ---
try:
    from kaggle_secrets import UserSecretsClient
    print("🔑 Detected Kaggle environment. Attempting to load secrets...")
    user_secrets = UserSecretsClient()
    all_found_kaggle = True
    for key in REQUIRED_SECRETS:
        try:
            secrets[key] = user_secrets.get_secret(key)
            if not secrets[key]:
                print(f"  ⚠️ Secret '{key}' found in Kaggle but is empty.")
                # Decide if empty is acceptable or an error
                # For critical keys, treat empty as an error
                if key in ["GOOGLE_API_KEY", "REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET"]:
                    raise ValueError(f"Critical Kaggle secret '{key}' is empty.")
            else:
                 print(f"  ✅ Loaded '{key}' from Kaggle.")
        except Exception as e:
            print(f"  ❌ Failed to load '{key}' from Kaggle Secrets: {e}")
            all_found_kaggle = False
            # Don't break, allow trying other methods if preferred,
            # but we'll likely fail later if critical keys are missing.

    if all_found_kaggle and all(secrets.values()): # Check if all keys have non-empty values
        secrets_loaded_successfully = True
        secret_source = "Kaggle"

except ImportError:
    print("🤷 Kaggle environment not detected.")
except Exception as e:
    print(f"❌ Unexpected error during Kaggle secret loading: {e}")

# --- Attempt 2: Google Colab Secrets (if Kaggle failed or wasn't detected) ---
if not secrets_loaded_successfully:
    try:
        from google.colab import userdata
        print("🔑 Detected Google Colab environment. Attempting to load secrets...")
        all_found_colab = True
        for key in REQUIRED_SECRETS:
            try:
                # Use userdata.get for graceful handling if key doesn't exist
                value = userdata.get(key)
                if value:
                    secrets[key] = value
                    print(f"  ✅ Loaded '{key}' from Colab.")
                else:
                    print(f"  ❌ Secret '{key}' not found or empty in Colab Secrets.")
                    all_found_colab = False
                    # Break early if a critical key is missing in Colab
                    if key in ["GOOGLE_API_KEY", "REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET"]:
                         raise ValueError(f"Critical Colab secret '{key}' not found or empty.")
            except Exception as e:
                 print(f"  ❌ Error loading '{key}' from Colab Secrets: {e}")
                 all_found_colab = False
                 break # Stop trying Colab secrets if one errors badly

        if all_found_colab and all(secrets.values()):
             secrets_loaded_successfully = True
             secret_source = "Google Colab"

    except ImportError:
        print("🤷 Google Colab environment not detected.")
    except Exception as e:
        print(f"❌ Unexpected error during Colab secret loading: {e}")


# --- Attempt 3: Local Environment Variables / .env file (if others failed) ---
if not secrets_loaded_successfully:
    print("🔑 Attempting to load secrets from local environment variables or .env file...")
    # Attempt to load .env file if it exists (useful for local dev)
    # Create a file named '.env' in the same directory as your script/notebook
    # Add lines like: GOOGLE_API_KEY="your_key_here"
    load_dotenv() # Loads variables from .env into os.environ

    all_found_local = True
    for key in REQUIRED_SECRETS:
        value = os.environ.get(key)
        if value:
            secrets[key] = value
            print(f"  ✅ Loaded '{key}' from environment.")
        else:
            print(f"  ❌ Secret '{key}' not found in environment variables (or .env file).")
            all_found_local = False
            # Break early if critical local env var is missing
            if key in ["GOOGLE_API_KEY", "REDDIT_CLIENT_ID", "REDDIT_CLIENT_SECRET"]:
                 break

    if all_found_local and all(secrets.values()):
        secrets_loaded_successfully = True
        secret_source = "Local Environment/.env"

# --- Final Check and Assignment ---
if secrets_loaded_successfully:
    # Assign secrets to global variables for other cells to use
    GOOGLE_API_KEY = secrets["GOOGLE_API_KEY"]
    REDDIT_CLIENT_ID = secrets["REDDIT_CLIENT_ID"]
    REDDIT_CLIENT_SECRET = secrets["REDDIT_CLIENT_SECRET"]
    REDDIT_USER_AGENT = secrets["REDDIT_USER_AGENT"]
    # Assign others if they were required (e.g., username/password)
    # REDDIT_USERNAME = secrets.get("REDDIT_USERNAME")
    # REDDIT_PASSWORD = secrets.get("REDDIT_PASSWORD")
    print(f"✅ Successfully loaded secrets via: {secret_source}")
else:
    print("❌ CRITICAL ERROR: Failed to load required secrets from any source (Kaggle, Colab, Environment).")
    print("   Please ensure secrets are correctly configured for your environment:")
    print("   - Kaggle: Add secrets via 'Add-ons' -> 'Secrets'.")
    print("   - Colab: Add secrets via the 'Secrets' tab (🔑 icon).")
    print("   - Local: Set environment variables OR create a '.env' file.")
    # Set flags to prevent API initialization later
    GOOGLE_API_KEY = None
    REDDIT_CLIENT_ID = None
    # etc.

print("🔐 Secrets Fetched & Loaded.")

In [ ]:
# Cell 3: API Initialization
# --------------------------
# Initializes the PRAW (Reddit) and Gemini clients using the secrets.
# loaded in the previous cell. Sets global variables 'reddit' and 'model'.

# Ensure necessary libraries were imported in Cell 2
import praw
import google.generativeai as genai

# --- Initialize Global API Client Variables ---
reddit = None # PRAW client instance
model = None  # Gemini client instance
initialization_error_message = None
apis_initialized_successfully = False # Flag to check before launching UI

print("🚀 Attempting to initialize API clients...")

# Check if secrets were loaded successfully in Cell 2 before proceeding
# Assumes GOOGLE_API_KEY, REDDIT_CLIENT_ID etc. are global vars set in Cell 2
if 'secrets_loaded_successfully' in globals() and secrets_loaded_successfully:
    try:
        # --- Initialize PRAW (Reddit Client) ---
        # Uses the credentials loaded from secrets.
        # This version assumes 'script' type app credentials (ID, Secret, User Agent)
        # and does NOT require username/password for basic read/search operations.
        print("  🔗 Initializing PRAW (Reddit client)...")
        reddit = praw.Reddit(
            client_id=REDDIT_CLIENT_ID,
            client_secret=REDDIT_CLIENT_SECRET,
            user_agent=REDDIT_USER_AGENT,
            # read_only=True # Can set True if only reading data, might avoid some warnings
        )
        # Perform a basic check to confirm connection was established.
        # Accessing reddit.read_only doesn't require authentication beyond app credentials.
        print(f"    ✅ PRAW instance created. Read-only status: {reddit.read_only}")

        # --- Configure Google Generative AI (Gemini Client) ---
        # Uses the Google API Key loaded from secrets.
        print("  ✨ Configuring Google Generative AI (Gemini client)...")
        genai.configure(api_key=GOOGLE_API_KEY)

        # Specify the Gemini model to use (e.g., 'gemini-pro', 'gemini-2.0-flash')
        # Using the latest one we settled on previously
        model_name = 'gemini-2.5-pro-exp-03-25'
        model = genai.GenerativeModel(model_name)

        # Optional: Verify model initialization by making a tiny test call (uncomment if needed)
        # try:
        #     _ = model.generate_content("test", generation_config=genai.types.GenerationConfig(max_output_tokens=5))
        #     print(f"    ✅ Gemini model '{model.model_name}' initialized and tested.")
        # except Exception as test_e:
        #     print(f"    ⚠️ Gemini model '{model.model_name}' initialized, but test call failed: {test_e}")
        #     # Decide if this is critical - maybe API key is valid but has no quota?
        print(f"    ✅ Gemini model '{model.model_name}' initialized.")


        # If both initializations succeeded without throwing errors:
        apis_initialized_successfully = True
        print("\n✅✅ API clients initialized successfully!")

    except praw.exceptions.PRAWException as praw_e:
        # Catch specific PRAW errors during initialization
        initialization_error_message = f"❌ PRAW Initialization Error: {praw_e}"
        print(initialization_error_message)
        reddit = None # Ensure client is None on failure
    except Exception as e:
        # Catch other errors (e.g., from Gemini initialization)
        initialization_error_message = f"❌ Error during API client initialization: {e}"
        print(initialization_error_message)
        # Reset both clients if one fails during setup
        reddit = None
        model = None
else:
    # This message comes from Cell 2 if secrets failed to load
    initialization_error_message = "❌ Cannot initialize APIs because secrets were not loaded successfully."
    print(initialization_error_message)

print("🚀 API Initialization Completed.")

In [ ]:
# Cell 4: Define Find Posts Function
# ----------------------------------
# Defines the main function for searching Reddit posts based on various criteria.
# Includes subreddit validation, typo suggestions, and negative keyword filtering.

import praw # Ensure praw is imported
import prawcore # Import prawcore for specific exceptions
import pandas as pd # Ensure pandas is imported

print("🛠️ Defining function: find_relevant_posts...")

def find_relevant_posts(keywords, negative_keywords_str, subreddit_list_str, min_score, min_comments, time_filter, sort_order):
    """
    Searches Reddit based on inputs including negative keywords, validates subreddits,
    suggests typo fixes (formatted as bullets with emojis), and returns a pandas DataFrame and status string.
    Requires the global 'reddit' PRAW client to be initialized.

    Args:
        keywords (str): Primary search terms (supports boolean operators).
        negative_keywords_str (str): Comma-separated terms to exclude.
        subreddit_list_str (str): Comma-separated list of subreddit names.
        min_score (int): Minimum upvote score for posts.
        min_comments (int): Minimum number of comments for posts.
        time_filter (str): Time period for search ('Hour', 'Day', 'Week', etc.).
        sort_order (str): Sorting method ('Relevance', 'Hot', 'Top', etc.).

    Returns:
        tuple: (pandas.DataFrame containing results, str containing status messages)
    """
    global reddit # Use the globally initialized PRAW client
    if not reddit:
        # Handle case where PRAW client wasn't initialized successfully
        return pd.DataFrame(), "❌ Error: Reddit client not initialized. Check Cell 3."

    # Initialize display_df to an empty DataFrame upfront.
    # This prevents an UnboundLocalError if no posts are found at all.
    display_df = pd.DataFrame(columns=["Subreddit", "Title", "Score", "Comments", "URL", "ID"])

    # --- Input Validation ---
    if not keywords or not subreddit_list_str:
        return display_df, "⚠️ Error: Please provide keywords and at least one subreddit."
    # Clean and split subreddit list
    target_subreddits = [sub.strip() for sub in subreddit_list_str.split(',') if sub.strip()]
    if not target_subreddits:
         return display_df, "⚠️ Error: Invalid subreddit list (make sure they are comma-separated)."

    # --- Process Negative Keywords ---
    # Start with the positive keywords, removing leading/trailing whitespace
    final_query = keywords.strip()
    # Check if negative keywords were provided
    if negative_keywords_str and negative_keywords_str.strip():
        # Clean and split negative keywords
        negative_keywords = [neg.strip() for neg in negative_keywords_str.split(',') if neg.strip()]
        if negative_keywords:
            # Format for PRAW search query using Lucene syntax: NOT (term1 OR term2)
            # Quote phrases containing spaces
            not_clause = " OR ".join([f'"{kw}"' if ' ' in kw else kw for kw in negative_keywords])
            # Append the NOT clause to the main query
            if final_query: # Ensure space if combining with positive keywords
                 final_query += f" NOT ({not_clause})"
            else: # Handle edge case of only negative keywords (might not be very useful)
                 final_query = f"NOT ({not_clause})"
    print(f"🔎 Using search query: {final_query}")
    # --- End Negative Keywords Processing ---

    all_potential_posts = {} # Dictionary to store unique found posts (using ID as key)
    validation_results = [] # List to store status messages for each subreddit
    search_info = f"⏳ Starting search across {len(target_subreddits)} subreddits...\n---\n"
    print(search_info)

    valid_sub_count = 0 # Counter for accessible subreddits

    # --- Loop through each target subreddit ---
    for sub_name in target_subreddits:
        # Format status message line with Markdown bullet
        current_sub_status = f"* `r/{sub_name}`: "
        print(f"  -> Processing r/{sub_name}...")
        try:
            # --- Subreddit Validation ---
            # Attempt to get the subreddit object. This implicitly validates it.
            subreddit = reddit.subreddit(sub_name)
            _ = subreddit.display_name # Access an attribute to force validation check

            # --- Perform Search within the valid subreddit ---
            search_results = subreddit.search(
                final_query, # Use the potentially modified query
                sort=sort_order.lower(),
                time_filter=time_filter.lower(),
                syntax='lucene', # Use Lucene syntax for NOT operator support
                limit=50 # Limit results per subreddit for performance
            )

            # --- Process Search Results ---
            count = 0 # Counter for relevant posts found in this subreddit
            for submission in search_results:
                # Check if post already found (avoid duplicates if searching overlapping subs)
                if submission.id not in all_potential_posts:
                    # Apply score and comment filters
                    if submission.score >= min_score and submission.num_comments >= min_comments:
                         all_potential_posts[submission.id] = submission
                         count += 1
            current_sub_status += f"✅ OK (Found {count} relevant posts)."
            valid_sub_count += 1

        # --- Handle Specific PRAW/Reddit Errors ---
        except prawcore.exceptions.NotFound:
             # Subreddit doesn't exist or is private/banned
             current_sub_status += "❌ Not Found."
             # Attempt to find typo suggestions
             try:
                 print(f"    -> '{sub_name}' not found, searching for suggestions...")
                 # Use search_by_name (without limit argument - fixed)
                 suggestions = list(reddit.subreddits.search_by_name(sub_name, exact=False))
                 if suggestions:
                     # Format suggestions nicely, limit to 3
                     suggested_names = [f"`r/{s.display_name}`" for s in suggestions[:3]]
                     current_sub_status += f" Did you mean: {', '.join(suggested_names)}?"
                 else:
                     current_sub_status += " (No suggestions found)."
             except Exception as sugg_e:
                 # Handle errors during the suggestion search itself
                 print(f"    -> Error searching suggestions for '{sub_name}': {sugg_e}")
                 current_sub_status += " (Suggestion search failed)."

        except prawcore.exceptions.Forbidden:
             # Access denied (e.g., private sub, user banned from sub)
             current_sub_status += "🚫 Forbidden."
        except prawcore.exceptions.Redirect:
             # Subreddit name might have changed or other redirect issue
             current_sub_status += f"↪️ Redirected (Check spelling/name for `r/{sub_name}`)."
        except prawcore.exceptions.PrawcoreException as core_e:
             # Catch other general PRAW core API errors
             current_sub_status += f"⚠️ PRAW Core Error: {core_e}"
        except Exception as e:
            # Catch any other unexpected errors during processing
            current_sub_status += f"⚠️ Error: {e}"

        # Store the status message for this subreddit
        validation_results.append(current_sub_status)
        # Print clean status to console
        print(f"    -> Status: {current_sub_status.replace('* `','').replace('`','')}")


    # --- Compile Final Results and Status ---
    # Format the validation summary as a Markdown bulleted list
    validation_summary = "\n---\n**Subreddit Status:**\n" + "\n".join(validation_results)
    post_data = [] # List to hold data for the DataFrame

    # If any posts were found across all valid subreddits
    if all_potential_posts:
        # Sort posts by score (descending)
        sorted_posts = sorted(all_potential_posts.values(), key=lambda p: p.score, reverse=True)
        # Extract relevant data for the DataFrame
        for post in sorted_posts:
             post_data.append({
                 "Subreddit": f"r/{post.subreddit.display_name}", "Title": post.title,
                 "Score": post.score, "Comments": post.num_comments,
                 "URL": f"https://www.reddit.com{post.permalink}", "ID": post.id
             })
        # Create the DataFrame from the collected data
        df_results = pd.DataFrame(post_data)
        # Update display_df (initialized empty earlier) with the results (limited to 25)
        display_df = df_results.head(25)
        # Create the final status message for successful search
        final_status_msg = f"✅ Search complete ({valid_sub_count} accessible subreddits).\nDisplaying {len(display_df)} posts."
    else:
        # If no posts were found, display_df remains empty
        final_status_msg = f"🏁 Search complete ({valid_sub_count} accessible subreddits).\nNo posts matched all criteria."

    # Combine the overall status with the detailed subreddit validation summary
    full_status_message = final_status_msg + validation_summary
    # Return the DataFrame (potentially empty) and the full status message string
    return display_df, full_status_message

print("✅ Function 'find_relevant_posts' defined.")

In [ ]:
# Cell 4.5: Define Recommend Subreddits Function
# ---------------------------------------------
# Defines the logic for recommending subreddits based on keywords.

import praw # Ensure praw is imported
import prawcore # Ensure prawcore is imported

print("🛠️ Defining function: recommend_subreddits...")

def recommend_subreddits(keywords):
    """
    Searches for subreddits related to the given keywords using PRAW's
    subreddit search functionality.

    Args:
        keywords (str): Keywords to search for in subreddit names/descriptions.

    Returns:
        str: A formatted string with recommendations or an error message.
             Requires the global 'reddit' PRAW client to be initialized.
    """
    global reddit # Use the globally initialized PRAW client
    if not reddit:
        return "❌ Error: Reddit client not initialized. Check Cell 3."
    if not keywords:
        return "⚠️ Error: Please enter keywords to get recommendations."

    print(f"🔎 Searching for subreddits related to: '{keywords}'")
    recommendations = [] # List to store formatted recommendation strings
    try:
        # Use reddit.subreddits.search() to find matching subreddits
        # limit=10 restricts the number of results returned by the API
        results = reddit.subreddits.search(keywords, limit=10)

        # Iterate through the Subreddit objects returned
        for sub in results:
            # Format the output string for each recommendation
            # Includes display name and subscriber count (formatted with commas)
            recommendations.append(f"- `r/{sub.display_name}` ({sub.subscribers:,} subscribers)")

        # Check if any recommendations were found
        if recommendations:
            # Return a formatted string with the list of recommendations
            return f"💡 Found potential subreddits related to '{keywords}':\n" + "\n".join(recommendations)
        else:
            # Return a message if no subreddits matched the keywords
            return f"🤔 No subreddit recommendations found for '{keywords}'. Try broader terms."

    # --- Handle Potential Errors during Subreddit Search ---
    except prawcore.exceptions.PrawcoreException as core_e:
         # Catch specific PRAW core errors
         error_msg = f"❌ Error searching for subreddits: {core_e}"
         print(f"  - {error_msg}")
         return error_msg
    except Exception as e:
        # Catch any other unexpected errors
        error_msg = f"❌ An unexpected error occurred: {e}"
        print(f"  - {error_msg}")
        return error_msg

print("✅ Function 'recommend_subreddits' defined.")

In [ ]:
# Cell 5: Define Get Suggestions Function
# ---------------------------------------
# Defines the main function for fetching post/comment context and generating
# AI suggestions using the Gemini model. Includes AI Tone customization.

import time
import praw # Ensure praw is imported
import prawcore # Ensure prawcore is imported
from urllib.parse import urlparse # Ensure urlparse is imported
import google.generativeai as genai # Ensure genai is imported

print("🛠️ Defining function: get_ai_suggestions...")

# Added tone_style parameter with a default value
def get_ai_suggestions(post_identifier, user_expertise, tone_style="Default"):
    """
    Gets AI comment suggestions, placement advice, and full post details for preview.
    Accepts a tone_style parameter to guide the AI. Fetches top 10 comments. Uses improved prompt.
    Requires global 'reddit' PRAW client and 'model' Gemini client to be initialized.

    Args:
        post_identifier (str): The URL or ID of the target Reddit post.
        user_expertise (str): Text describing the user's area of knowledge/service.
        tone_style (str): The desired tone for the AI suggestions (e.g., "Casual", "Formal").

    Returns:
        tuple: (suggestions, post_info, top_comments_formatted, placement_advice, post_preview)
               Contains AI suggestions, processing status, formatted comments, placement advice,
               and a dictionary with post preview details. Returns error messages on failure.
    """
    global reddit, model # Use globally initialized clients
    # Define default dictionary for preview in case of errors
    error_preview = {"title": "Error", "author": "", "text": "Could not fetch post details.", "url": ""}
    # Define default return tuple for error cases
    error_return = ("Error processing request.", "Error", "N/A", "N/A", error_preview)

    # --- Input Validation & Prerequisite Checks ---
    if not model: return ("❌ Error: Gemini model not initialized. Check Cell 3.", "", "", "", error_preview)
    if not reddit: return ("❌ Error: Reddit client not initialized. Check Cell 3.", "", "", "", error_preview)
    if not post_identifier: return ("⚠️ Error: Please provide a Reddit Post URL or ID.", "", "", "", error_preview)
    if not user_expertise: return ("⚠️ Error: Please provide your area of expertise.", "", "", "", error_preview)

    # Initialize variables
    submission = None
    post_info = ""
    top_comments_formatted = "Could not fetch comments." # Default message for display
    top_comments_for_prompt = "Could not fetch comments." # Default for prompt context
    placement_advice = "Could not generate placement advice."
    post_preview = {"title": "Fetching...", "author": "", "text": "", "url": ""}

    print(f"🚀 Attempting to get suggestions for: {post_identifier} with tone: {tone_style}")

    try:
        # --- Fetch the specific Reddit submission ---
        post_id = None
        # Determine if input is a URL or just an ID
        if "reddit.com" in post_identifier:
            try: # Parse URL to extract post ID
                parsed = urlparse(post_identifier); path_parts = [p for p in parsed.path.split('/') if p]
                if len(path_parts) >= 4 and path_parts[2] == 'comments': post_id = path_parts[3]
                else: raise ValueError("Could not parse post ID from URL structure.")
            except Exception as url_e: return f"❌ Error parsing URL: {url_e}", "", "", "", error_preview
        else: # Assume input is a raw ID
             post_id = post_identifier.strip()

        if not post_id: return "❌ Error: Failed to determine Post ID.", "", "", "", error_preview

        print(f"  🔗 Fetching submission object for ID: {post_id}...")
        submission = reddit.submission(id=post_id) # Fetch submission object using PRAW

        # Populate preview dictionary with fetched data
        post_preview = {
            "title": submission.title,
            "author": f"u/{submission.author.name}" if submission.author else "[deleted]", # Handle deleted authors
            "url": f"https://www.reddit.com{submission.permalink}", # Construct full URL
            # Get full selftext for text posts, or indicate it's a link post
            "text": submission.selftext if submission.is_self else f"Link Post: {submission.url}"
        }
        # Create status message about the processed post
        post_info = f"✅ Processed Post: '{submission.title}' (r/{submission.subreddit.display_name})"
        print(f"    -> Post Found: '{submission.title}'")

        # --- Fetch Top 10 Comments ---
        print("  💬 Fetching top 10 comments...")
        fetched_comments_display = [] # For displaying in UI
        comments_for_prompt_context = [] # For sending to AI
        try:
            # Load comment forest and replace "MoreComments" objects
            submission.comments.replace_more(limit=0)
            comment_limit = 10 # Set desired number of comments

            # Iterate through the top-level comments
            for i, top_level_comment in enumerate(submission.comments.list()[:comment_limit]):
                 # Check if comment body exists and is valid
                 if hasattr(top_level_comment, 'body') and top_level_comment.body not in ['[deleted]', '[removed]']:
                     author = top_level_comment.author.name if top_level_comment.author else '[deleted]'
                     # Create excerpt for display and prompt (limit length)
                     body_excerpt = top_level_comment.body[:250].replace('\n', ' ') + ('...' if len(top_level_comment.body) > 250 else '')
                     # Format for display in UI (includes score)
                     comment_display_str = f"{i+1}. u/{author} (Score: {top_level_comment.score}): {body_excerpt}"
                     fetched_comments_display.append(comment_display_str)
                     # Format for AI prompt context (simpler, no score)
                     comments_for_prompt_context.append(f"{i+1}. u/{author}: {body_excerpt}")

            # Update formatted strings if comments were found
            if fetched_comments_display:
                top_comments_formatted = "\n\n".join(fetched_comments_display) # Add extra newline for UI readability
                top_comments_for_prompt = "\n".join(comments_for_prompt_context)
                print(f"    -> Fetched {len(fetched_comments_display)} top comments.")
            else:
                 top_comments_formatted = "🤔 No suitable top-level comments found."
                 top_comments_for_prompt = "No comments available."
                 print("    -> No suitable top-level comments found.")
        except Exception as comment_e:
            # Handle errors during comment fetching
            print(f"    -> ⚠️ Warning: Could not fetch comments: {comment_e}")
            top_comments_formatted = f"⚠️ Could not fetch comments: {comment_e}"
            top_comments_for_prompt = "Could not fetch comments."

        # --- Prepare context and call Gemini API with IMPROVED Prompt including TONE ---
        print(f"  🤖 Preparing prompt (tone: {tone_style}) and calling Gemini API...")
        # Use the full post text/link from preview, limit length for safety
        post_context_for_prompt = post_preview['text'][:3000] + ("...\n[Post Content Truncated]" if len(post_preview['text']) > 3000 else "")
        # Combine all context elements for the prompt
        context = f"Reddit Post Title: {post_preview['title']}\nReddit Post Content/Link: {post_context_for_prompt}\n\nTop Comments Context (up to 10):\n---\n{top_comments_for_prompt}\n---"

        # --- Significantly Improved Prompt ---
        # Includes instructions for tone, number of suggestions, placement advice, and desired quality.
        prompt = f"""
        Analyze the provided Reddit post, its top comments, and my area of expertise to generate insightful engagement opportunities.

        My Area of Expertise:
        ---
        {user_expertise}
        ---

        Desired Comment Tone: **{tone_style if tone_style != "Default" else "Neutral/Helpful"}**

        Reddit Post & Top Comments Context:
        ---
        {context}
        ---

        **Task:**

        1.  **Generate 4-5 Distinct Comment Ideas:** Create suggestions that match the **Desired Comment Tone**. They must be:
            * Highly Relevant & Value-Adding: Directly address points in the post or comments. Offer unique insights, helpful explanations, relevant experiences, or thoughtful questions related to my expertise AND the ongoing discussion.
            * Context-Aware: Show understanding of the existing comments. Avoid repeating points.
            * Non-Promotional & Constructive: Focus on contributing positively. No calls to action or direct self-promotion.
            * Varied Angles (analytical, practical, supportive, etc.) fitting the tone.
            * Actionable Starting Points (2-4 sentences). Number them clearly.

        2.  **Provide Placement Advice:** Under a heading "Placement Advice:", recommend THE BEST STRATEGY (new top-level comment OR reply to a specific numbered comment/author from the context). Explain your reasoning briefly.

        **Output Format:**
        First, list the numbered comment ideas.
        Second, provide the "Placement Advice:" section.
        """

        # Make the API call to Gemini
        response = model.generate_content(prompt)

        # --- Process and return response ---
        if response.text:
             full_response = response.text.strip()
             suggestions = full_response # Default if parsing fails
             placement_advice = "Placement advice not found or failed to parse." # Default

             # Attempt to split the response into suggestions and advice
             split_keyword = "Placement Advice:"
             parts = full_response.split(split_keyword, 1)
             if len(parts) == 2:
                 suggestions = parts[0].strip()
                 # Clean up potential leading/trailing list formatting noise from suggestions
                 suggestions = '\n'.join(line.strip() for line in suggestions.splitlines() if line.strip())
                 placement_advice = parts[1].strip()
             else: # Try alternative keyword if primary one fails
                  split_keyword_alt = "Recommendation:"
                  parts = full_response.split(split_keyword_alt, 1)
                  if len(parts) == 2:
                     suggestions = parts[0].strip()
                     suggestions = '\n'.join(line.strip() for line in suggestions.splitlines() if line.strip())
                     placement_advice = parts[1].strip()
                  else: # Log warning if parsing failed
                     print("⚠️ Warning: Could not clearly parse 'Placement Advice:' from AI response.")

             print("    -> Suggestions and advice generated successfully.")
             # Return all 5 pieces of information
             return suggestions, post_info, top_comments_formatted, placement_advice, post_preview
        else: # Handle cases where the response might be blocked or empty
             block_reason = response.prompt_feedback.block_reason if response.prompt_feedback else 'Unknown'
             error_msg = f"❌ Error: Could not generate suggestions. Response blocked/empty. Reason: {block_reason}"
             print(f"    -> {error_msg}")
             # Return error message in suggestions field, but other fetched info if available
             return error_msg, post_info, top_comments_formatted, placement_advice, post_preview

    # --- Handle Errors During Post Fetching ---
    except prawcore.exceptions.NotFound:
         # Specific error if the post ID itself is invalid
         error_msg = f"❌ Error: Reddit post with identifier '{post_identifier}' not found."
         print(f"  - {error_msg}")
         return error_msg, "Post Not Found", "N/A", "N/A", error_preview
    except Exception as e:
        # Catch any other unexpected errors during the process
        error_msg = f"❌ Error processing post or getting suggestions: {e}"
        print(f"  - {error_msg}")
        # Update preview text to show the error occurred
        post_preview["text"] = f"Error occurred: {e}"
        # Return error message in suggestions field, potentially partial info elsewhere
        return error_msg, post_info, top_comments_formatted, placement_advice, post_preview

print("✅ Function 'get_ai_suggestions' defined.")

In [ ]:
# Cell 6: Define and Launch Gradio UI
# -----------------------------------
# Defines the Gradio web interface layout, connects UI elements to backend functions,
# and launches the application. Includes all features and uses the Soft theme with Inter font.

import gradio as gr
import pandas as pd # Ensure pandas is imported
# Import GoogleFont theme component if not already imported globally
from gradio.themes.utils import colors, fonts, sizes

print("🎨 Defining Gradio UI...")

# --- Prerequisite Check ---
# Verify that the API clients were initialized successfully in Cell 3
if 'apis_initialized_successfully' not in globals() or not apis_initialized_successfully:
    print("\n🛑 Cannot build Gradio UI because API clients failed to initialize in Cell 3.")
    # Display previous error messages if they exist
    if 'initialization_error_message' in globals() and initialization_error_message: print(initialization_error_message)
    if 'kaggle_secrets_error_message' in globals() and kaggle_secrets_error_message: print(kaggle_secrets_error_message)
    # Optionally, raise an error or exit if running as a script
    # raise RuntimeError("API clients not initialized. Cannot launch UI.")

else:
    # --- Custom CSS ---
    # Minimal CSS for font size increase and basic button hover effect.
    custom_css = """
    /* Slightly larger base font (relative to the theme's base) */
    body, .gradio-container, .gradio-container button, .gradio-container input, .gradio-container textarea, .gradio-container select {
        font-size: 103% !important;
    }
    /* Basic button hover effect for light theme */
     #find_posts_button:hover, #get_suggestions_button:hover, #recommend_subs_button:hover, #copy_suggestion_button:hover {
        box-shadow: 0 2px 5px rgba(0,0,0,0.1); /* Standard shadow */
        filter: brightness(0.98); /* Slightly darken */
        transform: translateY(-1px);
        transition: all 0.15s ease-in-out;
    }
    """

    # --- Wrapper Functions for Loading State ---
    # These functions handle UI updates (like disabling buttons and showing "Loading...")
    # while the main backend functions are running. They use 'yield' to update the UI incrementally.

    def find_posts_wrapper(keywords, negative_keywords_str, subreddit_list_str, min_score, min_comments, time_filter, sort_order):
        """Wraps find_relevant_posts to show loading state."""
        print("UI: find_posts_wrapper triggered.")
        # Yield 1: Update UI to show loading and disable buttons
        yield {
            out_search_status: gr.Markdown("⏳ Searching Reddit & Validating Subreddits... Please wait."),
            btn_find_posts: gr.Button(interactive=False), # Disable self
            btn_recommend_subs: gr.Button(interactive=False) # Disable other button
        }
        # Call the actual backend function (defined in Cell 4)
        # Ensure find_relevant_posts is defined in the global scope
        df, status_msg = find_relevant_posts(keywords, negative_keywords_str, subreddit_list_str, min_score, min_comments, time_filter, sort_order)
        # Yield 2: Update UI with results and re-enable buttons
        yield {
            out_posts_df: df, # Update DataFrame output
            out_search_status: gr.Markdown(status_msg), # Update status message
            btn_find_posts: gr.Button(interactive=True), # Re-enable self
            btn_recommend_subs: gr.Button(interactive=True) # Re-enable other button
        }

    def recommend_subs_wrapper(keywords):
        """Wraps recommend_subreddits to show loading state."""
        print("UI: recommend_subs_wrapper triggered.")
        # Yield 1: Update UI to show loading and disable buttons
        yield {
            out_recommend_status: gr.Markdown("⏳ Searching for subreddit recommendations..."),
            btn_recommend_subs: gr.Button(interactive=False),
            btn_find_posts: gr.Button(interactive=False)
         }
        # Call the actual backend function (defined in Cell 4.5)
        # Ensure recommend_subreddits is defined in the global scope
        recommendation_text = recommend_subreddits(keywords)
        # Yield 2: Update UI with results and re-enable buttons
        yield {
            out_recommend_status: gr.Markdown(recommendation_text), # Update recommendation output
            btn_recommend_subs: gr.Button(interactive=True),
            btn_find_posts: gr.Button(interactive=True)
         }

    def get_suggestions_wrapper(post_identifier, user_expertise, tone_style):
        """Wraps get_ai_suggestions to show loading state."""
        print("UI: get_suggestions_wrapper triggered.")
        # Yield 1: Update UI to show loading and disable button
        yield {
            out_suggestion_post_info: gr.Markdown("⏳ Fetching post, top 10 comments, generating suggestions/advice..."),
            out_suggestions: "Loading...",
            out_top_comments: "Loading...",
            out_placement_advice: "Loading...",
            post_preview_title: "Loading...",
            post_preview_author: "Loading...",
            post_preview_text: "Loading...",
            btn_get_suggestions: gr.Button(interactive=False)
        }
        # Call the actual backend function (defined in Cell 5)
        # Ensure get_ai_suggestions is defined in the global scope
        suggestions, post_info, top_comments, placement_advice, post_preview = get_ai_suggestions(post_identifier, user_expertise, tone_style)
        # Yield 2: Update UI with results and re-enable button
        yield {
            out_suggestions: suggestions,
            out_suggestion_post_info: gr.Markdown(post_info),
            out_top_comments: top_comments,
            out_placement_advice: placement_advice,
            post_preview_title: post_preview.get("title", "N/A"), # Use .get for safety
            post_preview_author: post_preview.get("author", "N/A"),
            post_preview_text: post_preview.get("text", "N/A"),
            btn_get_suggestions: gr.Button(interactive=True)
        }

    # --- Helper Functions for UI Interaction ---

    def move_selected_post_to_tab2(selection: gr.SelectData):
        """Callback function triggered when a row is selected in the DataFrame."""
        # selection.index is the row index, selection.value is the cell value clicked
        if selection.selected:
            selected_value = selection.value
            # Check if the clicked value looks like a URL or a short ID
            if isinstance(selected_value, str) and ("reddit.com" in selected_value or len(selected_value) < 15):
                 print(f"UI: DataFrame row selected. Moving value: {selected_value} to Tab 2 input.")
                 # Return an update instruction for the target Textbox
                 return gr.Textbox(value=selected_value)
            else:
                 # If user clicked on Title/Score etc., log it but don't update the input
                 print(f"UI: DataFrame row selected, but clicked value '{selected_value}' doesn't look like URL/ID.")
                 # Return an update instruction with no value change
                 return gr.Textbox()
        else:
            # Triggered on deselect or other events, do nothing
            return gr.Textbox()

    def copy_suggestion_to_editor(suggestion_text):
        """Copies text from the AI suggestions output to the final editor input."""
        print("UI: Copying suggestion to editor...")
        # Return an update instruction for the target Textbox
        return gr.Textbox(value=suggestion_text)

    # --- Define the Theme ---
    # Use the Soft theme and specify the Inter font via Google Fonts
    inter_theme = gr.themes.Soft(
        primary_hue="indigo",
        font=fonts.GoogleFont("Inter") # Load Inter font
    )

    # --- Build Gradio Interface using Blocks ---
    # Defines the structure and components of the web UI.
    with gr.Blocks(theme=inter_theme, title="Reddit AI Assistant 🤖", css=custom_css) as iface:
        # Main title and description for the application
        gr.Markdown("# Reddit AI Assistant 🤖")
        gr.Markdown("Find posts, get recommendations & suggestions, view context, edit.")

        # --- Define UI Components within Tabs ---
        with gr.Tabs(elem_id="tabs_area"):
            # --- Tab 1: Finding Posts ---
            with gr.TabItem("1. Find Posts & Subreddits"):
                gr.Markdown("### Enter Search Criteria 🔍")
                # Row for main search inputs
                with gr.Row():
                    # Left column for text inputs
                    with gr.Column(scale=2):
                        # Keywords input (mandatory)
                        inp_keywords = gr.Textbox( label="Keywords *", placeholder='e.g., (python OR java) AND "data science"', info="Use operators like AND, OR, NOT, and quotes \"\" for exact phrases.", elem_id="keywords_input" )
                        # Negative keywords input (optional)
                        inp_negative_keywords = gr.Textbox( label="Negative Keywords (Optional)", placeholder="e.g., job, hiring, course", info="Comma-separated terms to exclude.", elem_id="negative_keywords_input" )
                        # Subreddits input (mandatory)
                        inp_subreddits = gr.Textbox( label="Target Subreddits *", placeholder="e.g., datascience, python", info="Enter one or more subreddit names, comma-separated.", elem_id="subreddits_input" )
                    # Right column for filters
                    with gr.Column(scale=1):
                        inp_min_score = gr.Slider(label="Min Score", minimum=0, maximum=500, value=10, step=5, info="Minimum post upvotes.", elem_id="min_score_slider")
                        inp_min_comments = gr.Slider(label="Min Comments", minimum=0, maximum=200, value=5, step=1, info="Minimum post comments.", elem_id="min_comments_slider")
                        inp_time_filter = gr.Dropdown(label="Time Filter", choices=['Hour', 'Day', 'Week', 'Month', 'Year', 'All'], value='Week', info="Search within this period.", elem_id="time_filter_dropdown")
                        inp_sort_order = gr.Dropdown(label="Sort Order", choices=['Relevance', 'Hot', 'Top', 'New', 'Comments'], value='Relevance', info="How to sort results.", elem_id="sort_order_dropdown")
                # Row for action buttons
                with gr.Row():
                     btn_find_posts = gr.Button("Find Posts & Validate Subreddits", variant="primary", elem_id="find_posts_button")
                     btn_recommend_subs = gr.Button("Recommend Subreddits based on Keywords", elem_id="recommend_subs_button")
                # Output areas for status and recommendations
                out_search_status = gr.Markdown(value="Status: Ready", elem_id="search_status_output")
                out_recommend_status = gr.Markdown(elem_id="recommend_status_output")
                gr.Markdown("---") # Separator
                # Output area for search results table
                gr.Markdown("### Found Posts Results 💡")
                gr.Markdown("*(Click on a URL or ID cell in the table to select it for Step 2)*")
                out_posts_df = gr.DataFrame(label="Found Posts Table", headers=["Subreddit", "Title", "Score", "Comments", "URL", "ID"], interactive=True, wrap=True, elem_id="posts_dataframe")

            # --- Tab 2: Getting Suggestions and Editing ---
            with gr.TabItem("2. Get Suggestions & Edit Comment"):
                gr.Markdown("### A. Input Post, Expertise & Tone 💬")
                gr.Markdown("*(Post URL/ID can be auto-filled by selecting from the table in Tab 1)*")
                # Row for inputs needed for suggestions
                with gr.Row():
                     with gr.Column(scale=3): # Post identifier input (mandatory)
                        inp_post_identifier = gr.Textbox( label="Reddit Post URL or ID *", placeholder="Select row in Tab 1 or paste URL/ID", info="The specific post for suggestions.", elem_id="post_id_input" )
                     with gr.Column(scale=2): # User expertise input (mandatory)
                        inp_user_expertise = gr.Textbox( label="Your Area of Expertise *", lines=3, placeholder="Confirm/edit your service or expertise...", info="Helps the AI tailor suggestions.", elem_id="expertise_input" )
                     with gr.Column(scale=1): # AI Tone selection (optional, has default)
                        inp_ai_tone = gr.Dropdown( label="Desired AI Tone", choices=["Default", "Casual", "Formal", "Informative", "Analytical", "Persuasive", "Friendly", "Humorous", "Gen-Z"], value="Default", info="Select the tone for AI suggestions.", elem_id="ai_tone_dropdown" )
                # Button to trigger suggestion generation
                btn_get_suggestions = gr.Button("Get Preview, Suggestions & Advice", variant="primary", elem_id="get_suggestions_button")
                # Status message for suggestion process
                out_suggestion_post_info = gr.Markdown(value="Status: Ready", elem_id="suggestion_post_info")
                gr.Markdown("---") # Separator

                # Section for displaying the post preview
                gr.Markdown("### B. Post Preview 🔤")
                post_preview_title = gr.Markdown(value="*Post title will appear here*")
                post_preview_author = gr.Markdown(value="")
                post_preview_text = gr.Textbox(label="Post Text / URL", lines=10, interactive=False) # Shows full text
                gr.Markdown("---") # Separator

                # Section for displaying context, suggestions, and advice
                gr.Markdown("### C. Context, Suggestions & Advice 🧠")
                with gr.Row(): # Arrange outputs side-by-side
                    with gr.Column(scale=1): # Top comments output
                        gr.Markdown("#### Top Comments (up to 10)")
                        out_top_comments = gr.Textbox(label="Fetched Top Comments", lines=18, interactive=False)
                    with gr.Column(scale=1): # AI suggestions output
                        gr.Markdown("#### AI Suggestions (4-5 expected)")
                        out_suggestions = gr.Textbox(label="Suggestions Output", lines=15, interactive=True)
                        btn_copy_suggestion = gr.Button("⬇️ Copy Suggestion to Editor Below", scale=1) # Button to copy suggestion
                    with gr.Column(scale=1): # Placement advice output
                        gr.Markdown("#### Placement Advice")
                        out_placement_advice = gr.Markdown(value="*AI advice will appear here.*")
                gr.Markdown("---") # Separator

                # Section for the final comment editor
                gr.Markdown("### D. Edit Your Final Comment 🗨️")
                gr.Markdown("*(Use the suggestions above as a starting point, then edit thoroughly here before manually copying & posting on Reddit)*")
                inp_final_comment_editor = gr.Textbox( label="Final Comment Editor:", lines=8, placeholder="Paste a suggestion (or use button above) and refine your comment here...", interactive=True, elem_id="final_comment_editor" )

        # --- Connect Functions to UI Components ---
        # Defines what happens when buttons are clicked or inputs change.

        # Connect 'Find Posts' button click to its wrapper function
        btn_find_posts.click(
            fn=find_posts_wrapper, # The function to call
            # List of input components whose values are passed to the function
            inputs=[inp_keywords, inp_negative_keywords, inp_subreddits, inp_min_score, inp_min_comments, inp_time_filter, inp_sort_order],
            # List of output components that the function's return/yield values will update
            outputs=[out_posts_df, out_search_status, btn_find_posts, btn_recommend_subs]
        )
        # Connect 'Recommend Subreddits' button click to its wrapper function
        btn_recommend_subs.click(
            fn=recommend_subs_wrapper, inputs=[inp_keywords],
            outputs=[out_recommend_status, btn_recommend_subs, btn_find_posts]
        )
        # Connect 'Get Suggestions' button click to its wrapper function
        btn_get_suggestions.click(
            fn=get_suggestions_wrapper, inputs=[inp_post_identifier, inp_user_expertise, inp_ai_tone], # Include tone input
            outputs=[ out_suggestions, out_suggestion_post_info, out_top_comments, out_placement_advice,
                      post_preview_title, post_preview_author, post_preview_text, btn_get_suggestions ] # Match all outputs
        )
        # Connect DataFrame selection event to the helper function
        out_posts_df.select(
            fn=move_selected_post_to_tab2, inputs=None, # Input is implicit SelectData
            outputs=[inp_post_identifier] # Target component to update
        )
        # Connect 'Copy Suggestion' button click to its helper function
        btn_copy_suggestion.click(
            fn=copy_suggestion_to_editor, inputs=[out_suggestions], # Source component
            outputs=[inp_final_comment_editor] # Target component
            )


    # --- Launch the Gradio Interface ---
    print("\n🚀 Launching Gradio interface...")
    print("   If successful, a public URL (share=True) will appear below.")
    print("   This link is temporary and only works while this cell is running.")
    # share=True creates a public link (needed for Kaggle/Colab)
    # debug=True provides more detailed error logs in the console if the Gradio app encounters issues
    iface.launch(share=True, debug=True)

    print("\n✅ Gradio app launched. Keep this cell running to use the interface.")
    # Note: The notebook cell must remain running for the Gradio server to be active.